# 03 — Faithfulness Evaluation Harness

This notebook builds the machinery for the central question of the thesis: **when an
explainability method says "the model relied on *this* part of the signal," is that true?**
That property is called **faithfulness**, and measuring it needs a small pipeline of pieces
that fit together. We build that pipeline in **6 layers**, one at a time, so each piece can be
understood and sanity-checked on its own before the next depends on it:

1. **Region grid** — how the signal is divided into regions *(this notebook section)*
2. Perturbation — how a region is masked/removed
3. Deletion (and insertion) curves — how confidence moves as regions are removed
4. XAI relevance on the grid — TimeSHAP / IG / attention scores per region
5. CMI — the faithfulness score combining the above
6. Cross-model comparison — the same metric across Models 1–5

> This notebook section is **Layer 1 only**. No perturbation, deletion curves, CMI, or XAI
> code appears here yet — those are later layers.

## Layer 1 — The region grid

**What it is.** The ECG200 signal is 96 timesteps long. Rather than reason about every single
timestep in isolation, we chop the signal into a handful of contiguous **regions** — small
consecutive chunks of the timeline — and reason about whole regions. The **region grid** is
simply the rule that says *which timesteps belong to which region*.

**Why it must be the single source of truth.** Two different parts of the harness use the grid,
and they have to agree perfectly:

- the **XAI methods** (Layer 4) will report how much relevance each *region* received, and
- **CMI** (Layer 5) will *perturb* the signal one *region* at a time and watch the model.

If those two sides ever divided the signal even slightly differently — say one used regions of
size 4 where the other used size 5 — then "the relevance of region 7" and "the region 7 we
perturbed" would refer to different timesteps, and every faithfulness number downstream would
be silently wrong. So the grid is defined in exactly **one place**, `src/xai/regions.py`, and
everything imports it. This notebook never re-defines it; it only imports and inspects it.

**Region size (data-justified for ECG200).** Region size is set as a **percentage of the signal
length**, and we use two: **10% (primary — the data-justified anchor)** and **5% (secondary —
the fine grid)**. These sizes come from ECG200's *own* structure: the autocorrelation analysis
in [`01b_signal_scale_analysis`](01b_signal_scale_analysis.ipynb) found the signal's natural
correlation length is **~10 timesteps**, so a 10% region (≈9.6 ts) is about one coherent unit of
signal, and a 5% region (≈4.8 ts) is a finer resolution that still stays above the correlation
length. (An earlier draft inherited Šimić et al.'s 2.5% for the fine grid; we moved away from it
because on this short 96-point signal 2.5% is only ~2–3 timesteps — well below the natural scale,
so it fragments features. See `DECISIONS_LOG.md`.) Both sizes are exposed as named constants so
the whole harness shares one definition.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.data.preprocessing import load_ecg200
from src.xai.regions import (
    build_region_grid,
    describe_grid,
    REGION_SIZES,
    REGION_SIZE_PRIMARY_PCT,
    REGION_SIZE_SECONDARY_PCT,
)

SIGNAL_LENGTH = 96  # ECG200

grid_primary   = build_region_grid(SIGNAL_LENGTH, REGION_SIZE_PRIMARY_PCT)    # 10% — anchor
grid_secondary = build_region_grid(SIGNAL_LENGTH, REGION_SIZE_SECONDARY_PCT)  # 5%  — fine grid

print("region sizes in use:", REGION_SIZES)

### The actual regions for a length-96 signal

`describe_grid` reports how many regions each size produces and the exact region lengths — the
sanity check that the partition is what we intend. Look for: full coverage (region sizes sum to
96), and region sizes that differ by at most 1.

In [ ]:
for name, grid in [("primary (10%)", grid_primary), ("secondary (5%)", grid_secondary)]:
    d = describe_grid(grid)
    print(f"=== {name} ===")
    print(f"  n_regions      : {d['n_regions']}")
    print(f"  region sizes   : {d['size_counts']}  (size: how many regions)")
    print(f"  min/max/mean   : {d['min_size']} / {d['max_size']} / {d['mean_size']:.2f}")
    print(f"  total coverage : {d['total_coverage']}  (covers full signal: {d['covers_fully']})")
    print(f"  first 3 bounds : {grid.bounds[:3].tolist()}  ...  last: {grid.bounds[-1].tolist()}")
    print()

### Seeing the grid on a real signal

Below is one example ECG200 beat with the region boundaries drawn as vertical lines — the fine
5% grid on top, the coarse 10% anchor below. This makes concrete how the same signal gets
chunked at the two resolutions: many narrow regions vs. a few wide ones.

In [ ]:
X_test, _ = load_ecg200("test")
signal = X_test[0]                      # one example beat
t = np.arange(SIGNAL_LENGTH)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, (name, grid) in zip(axes, [("5% — fine grid", grid_secondary),
                                    ("10% — coarse grid (anchor)", grid_primary)]):
    ax.plot(t, signal, color="C0", linewidth=1.3, zorder=3)
    # Region boundaries: the start of every region, plus the final end.
    boundaries = np.append(grid.bounds[:, 0], grid.bounds[-1, 1])
    for b in boundaries:
        ax.axvline(b - 0.5, color="C3", linewidth=0.7, alpha=0.6, zorder=1)
    ax.set_title(f"{name}  —  {grid.n_regions} regions")
    ax.set_ylabel("z-scored amplitude")
axes[-1].set_xlabel("timestep")
fig.tight_layout()

FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "03_region_grid.png", dpi=150, bbox_inches="tight")
plt.show()

### What the grid gives us (and the non-integer note)

For the length-96 ECG200 signal:

- **Primary, 10% → 10 regions**, with sizes **9 or 10** (4 regions of size 9, 6 of size 10). This
  is the data-justified anchor: ≈ the signal's ~10-timestep autocorrelation length, so each
  region is about one coherent unit of signal.
- **Secondary, 5% → 20 regions**, with sizes **4 or 5** (4 regions of size 4, 16 of size 5). This
  is the fine grid — higher resolution, but still above the correlation length rather than
  fragmenting features (which is why we moved off 2.5%; see below).

**Why the regions are not all the same width.** A percentage of 96 is usually not a whole number
of timesteps: **5% of 96 is 4.8 timesteps**, not an integer, so regions *cannot* all be the same
size. We handle this deliberately with a **count-based near-equal split**: we fix the number of
regions at `round(100 / pct)` — 10 for 10%, 20 for 5% — and divide the 96 timesteps into that
many contiguous regions as evenly as possible, distributing the remainder so region sizes differ
by **at most 1**. The **realized average region size is exactly 4.8** timesteps (96 / 20),
faithful to the intended 5%; the individual regions are simply 4 or 5 timesteps to make that
average work out over whole timesteps. The 10% grid is analogous: an average of 9.6 timesteps,
realized as regions of 9 or 10.

The alternative — rounding the *region length* to a whole number — was rejected because it drifts
away from the target percentage. This drift is most severe for small percentages on a short
signal, which is exactly the reason **2.5% was rejected for ECG200**: 2.5% of 96 is 2.4, which
would round to 2–3-timestep regions sitting well below the ~10-timestep natural scale. Fixing the
count keeps every region as close to its target percentage as whole timesteps allow, tiles the
whole signal with no gaps or overlaps, and is **deterministic**: the same `(length,
region_size_pct)` always returns the identical grid, so the XAI side and the CMI side are
guaranteed to share it.